# 08: Paper Figures 5-7

**Cold-Start Social-Graph Leakage: figures generated from the result files**

This notebook draws three figures of the paper from the CSV files that earlier notebooks write:

- Fig. 5 (parameter sweeps) and Fig. 6 (scoring-rule sensitivity) from `sweeps/results_long.csv` (notebook 03),
- Fig. 7 (held-out-edge evaluation) from `inductive/inductive_long.csv` (notebook 07).

Run notebooks 03 and 07 in `full` mode first. The figures are written to `cold_start_leakage/figures/` as PDF and PNG files.

In [ ]:
!pip install -q numpy pandas matplotlib

In [ ]:
# --- repository-root anchor (added when notebooks moved into notebooks/) ---
# Notebooks live in notebooks/ but read and write cold_start_leakage/ at the
# repository root, so resolve that root explicitly instead of relying on cwd.
import os as _os_anchor
_here = _os_anchor.getcwd()
if _os_anchor.path.basename(_here) == 'notebooks':
    _os_anchor.chdir(_os_anchor.path.dirname(_here))
# --- end anchor ---
import os

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')
print("Result root:", ROOT)

In [ ]:
# Paper figures 5-7 (parameter sweeps, scoring-rule sensitivity, held-out-edge evaluation).
# Reads the CSV files written by notebooks 03 and 07 and writes PDF/PNG files to cold_start_leakage/figures/.
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = os.path.join(ROOT, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
                     "lines.linewidth": 1.2, "lines.markersize": 4, "pdf.fonttype": 42, "ps.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREEN, VERM, GREY = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#7F7F7F"   # Okabe-Ito palette
from scipy.stats import t as stats_t
# half-width of the 95% confidence interval of the mean, based on the t distribution
ci = lambda x: stats_t.ppf(0.975, len(x) - 1) * np.std(x, ddof=1) / math.sqrt(len(x))
def panel(ax, letter):
    ax.text(-0.16, 1.04, f"({letter})", transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")

res = pd.read_csv(os.path.join(ROOT, "sweeps", "results_long.csv"))
ind = pd.read_csv(os.path.join(ROOT, "inductive", "inductive_long.csv"))
ms = res[(res.block == "multiseed") & (res.model == "lightgcn")]
fr = res[res.block == "formula"].copy(); fr["formula_value"] = fr["formula_value"].astype(str)

# Figures are drawn at their printed width (6.3 in), so no text is smaller than 7 pt in the paper.
# ---- Figure 6: scoring-rule sensitivity ----
fig, axs = plt.subplots(1, 3, figsize=(6.3, 2.8), constrained_layout=True)
ax = axs[0]
for g, col, mk, lab in [("sbm", BLUE, "o", "SBM"), ("lfr", ORANGE, "s", "LFR")]:
    xs, ys, es = [], [], []
    for kg in [2, 3, 5, 10, 20]:
        d = ms[ms.graph_type == g] if kg == 5 else fr[(fr.swept_param == "k_gate") & (fr.formula_value == str(kg)) & (fr.graph_type == g)]
        xs.append(kg); ys.append(d.warm_prec.mean()); es.append(ci(d.warm_prec))
    ax.errorbar(xs, ys, yerr=es, color=col, marker=mk, capsize=2, label=f"{lab} warm ($h=15$)")
    ax.axhline(ms[ms.graph_type == g].cold_prec.mean(), color=col, ls="--", lw=0.9, label=f"{lab} cold ($h=0$)")
ax.set_xlabel(r"gate scale $k_{\mathrm{gate}}$"); ax.set_ylabel("precision@20"); ax.set_ylim(0, 0.62); ax.set_xticks([2, 3, 5, 10, 20])
ax.legend(frameon=False, loc="upper left", fontsize=7, ncol=1, handlelength=1.6); panel(ax, "a")
ax = axs[1]
variants = [("rational\n(nominal)", None), ("expon-\nential", ("gate", "exponential")), ("step", ("gate", "step")), ("max\nfusion", ("fusion", "max"))]
x = np.arange(len(variants)); w = 0.38
for j, (g, col, lab, hatch) in enumerate([("sbm", BLUE, "SBM", None), ("lfr", ORANGE, "LFR", "//")]):
    vals, errs = [], []
    for name, key in variants:
        d = ms[ms.graph_type == g] if key is None else fr[(fr.swept_param == key[0]) & (fr.formula_value == key[1]) & (fr.graph_type == g)]
        vals.append(d.warm_prec.mean()); errs.append(ci(d.warm_prec))
    ax.bar(x + (j - 0.5) * w, vals, w, yerr=errs, capsize=2, color=col, edgecolor="black", linewidth=0.4, hatch=hatch, label=lab, error_kw=dict(lw=0.6))
ax.set_xticks(x); ax.set_xticklabels([v[0] for v in variants], fontsize=7); ax.set_ylabel("warm precision@20 ($h=15$)"); ax.set_ylim(0, 0.095)
ax.set_xlabel("gate shape or fusion rule"); ax.legend(frameon=False, loc="upper center", ncol=2); panel(ax, "b")
ax = axs[2]
cv = [("nominal", None), (r"$\beta$=0.1", "pop_blend_0.1"), (r"$\beta$=0.3", "pop_blend_0.3"), (r"$\beta$=0.5", "pop_blend_0.5"), ("raw dot", "raw_dot_0.0")]
x = np.arange(len(cv))
for j, (g, col, lab, hatch) in enumerate([("sbm", BLUE, "SBM", None), ("lfr", ORANGE, "LFR", "//")]):
    vals, errs = [], []
    for name, key in cv:
        d = ms[ms.graph_type == g] if key is None else fr[(fr.swept_param == "cold_path") & (fr.formula_value == key) & (fr.graph_type == g)]
        vals.append(d.cold_prec.mean()); errs.append(ci(d.cold_prec))
    ax.bar(x + (j - 0.5) * w, vals, w, yerr=errs, capsize=2, color=col, edgecolor="black", linewidth=0.4, hatch=hatch, label=lab, error_kw=dict(lw=0.6))
    ax.axhline(ms[ms.graph_type == g].warm_prec.mean(), color=col, ls=":", lw=0.9)
ax.set_xticks(x); ax.set_xticklabels([c[0] for c in cv], fontsize=7, rotation=25, ha="right"); ax.set_ylabel("cold precision@20 ($h=0$)"); ax.set_ylim(0, 0.52)
ax.set_xlabel("cold-path scoring variant"); ax.legend(frameon=False, loc="upper center", ncol=2); panel(ax, "c")
fig.savefig(os.path.join(FIG_DIR, "fig_scoring_rules.pdf")); fig.savefig(os.path.join(FIG_DIR, "fig_scoring_rules.png"), dpi=200); plt.close(fig)

# ---- Figure 5: parameter sweeps ----
fea = res[res.block == "feature"]; den = res[res.block == "density"]
fig, axs = plt.subplots(1, 2, figsize=(6.3, 2.4), constrained_layout=True)
ax = axs[0]
for g, col, mk, lab in [("sbm", BLUE, "o", "SBM"), ("lfr", ORANGE, "s", "LFR")]:
    d = fea[fea.graph_type == g].groupby("homophily").cold_prec.agg(["mean", "std", "count"]).reset_index()
    ax.errorbar(d.homophily, d["mean"], yerr=stats_t.ppf(0.975, d["count"] - 1) * d["std"] / np.sqrt(d["count"]), color=col, marker=mk, capsize=2, label=lab)
ax.set_xlabel(r"feature homophily $\eta$"); ax.set_ylabel("cold precision@20 (FeatureGNN)"); ax.set_ylim(0, 0.2); ax.set_xticks([0.3, 0.5, 0.7, 0.9])
ax.legend(frameon=False); panel(ax, "a")
ax = axs[1]
d = den[den.graph_type == "sbm"]
for col, mk, metric, lab in [(GREEN, "o", "cold_recall", "cold recall@20"), (VERM, "s", "cold_prec", "cold precision@20")]:
    g2 = d.groupby("density_val")[metric].agg(["mean", "std", "count"]).reset_index()
    ax.errorbar(g2.density_val, g2["mean"], yerr=stats_t.ppf(0.975, g2["count"] - 1) * g2["std"] / np.sqrt(g2["count"]), color=col, marker=mk, capsize=2, label=lab)
ax.set_xlabel(r"intra-community edge probability $p_{\mathrm{in}}$ (SBM)"); ax.set_ylabel("value (LightGCN)"); ax.set_ylim(0, 1.05); ax.set_xticks([0.04, 0.06, 0.08])
ax.legend(frameon=False, loc="center right"); panel(ax, "b")
fig.savefig(os.path.join(FIG_DIR, "fig_sweeps.pdf")); fig.savefig(os.path.join(FIG_DIR, "fig_sweeps.png"), dpi=200); plt.close(fig)

# ---- Figure 7: held-out-edge evaluation ----
fig, axs = plt.subplots(1, 2, figsize=(6.3, 3.0), constrained_layout=True)
ax = axs[0]
order = [("sbm", "lightgcn"), ("sbm", "featuregnn"), ("sbm", "mf_bpr"), ("lfr", "lightgcn"), ("lfr", "featuregnn"), ("lfr", "mf_bpr")]
names = {"lightgcn": "LightGCN", "featuregnn": "FeatureGNN", "mf_bpr": "MF-BPR"}
x = np.arange(len(order)); w = 0.38
tr = [ind[(ind.graph == g) & (ind.model == m)].auc_trans for g, m in order]
te = [ind[(ind.graph == g) & (ind.model == m)].auc_ind for g, m in order]
ax.bar(x - w / 2, [t.mean() for t in tr], w, yerr=[ci(t) for t in tr], capsize=2, color=GREY, edgecolor="black", linewidth=0.4, label="training edges", error_kw=dict(lw=0.6))
ax.bar(x + w / 2, [t.mean() for t in te], w, yerr=[ci(t) for t in te], capsize=2, color=GREEN, edgecolor="black", linewidth=0.4, hatch="//", label="held-out edges", error_kw=dict(lw=0.6))
ax.axhline(0.5, color="black", ls=":", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([f"{g.upper()} {names[m]}" for g, m in order], fontsize=7, rotation=30, ha="right")
ax.set_ylabel("link-prediction AUC"); ax.set_ylim(0.4, 1.13); ax.legend(frameon=False, loc="upper center", ncol=2); panel(ax, "a")
ax = axs[1]
mk = {"lightgcn": "o", "featuregnn": "s", "mf_bpr": "^"}; colg = {"sbm": BLUE, "lfr": ORANGE}
off = {("sbm", "featuregnn"): (-62, 8), ("lfr", "featuregnn"): (8, -13), ("sbm", "mf_bpr"): (6, 4), ("lfr", "mf_bpr"): (6, 4), ("sbm", "lightgcn"): (6, -10), ("lfr", "lightgcn"): (6, 4)}
for g, m in order:
    d = ind[(ind.graph == g) & (ind.model == m)]
    ax.errorbar(d.recall20_ind.mean(), d.leak_recall.mean(), xerr=ci(d.recall20_ind), yerr=ci(d.leak_recall), color=colg[g], marker=mk[m], capsize=2, ms=5, ls="none")
    ax.annotate(f"{g.upper()} {names[m]}", (d.recall20_ind.mean(), d.leak_recall.mean()), xytext=off[(g, m)], textcoords="offset points", fontsize=7)
ax.plot([0, 1], [0, 1], color="black", ls=":", lw=0.8); ax.set_xlim(0, 0.35); ax.set_ylim(0, 1.0)
ax.set_xlabel("held-out recall@20 (recommendation utility)"); ax.set_ylabel("cold recall@20 (leakage)"); panel(ax, "b")
fig.savefig(os.path.join(FIG_DIR, "fig_heldout.pdf")); fig.savefig(os.path.join(FIG_DIR, "fig_heldout.png"), dpi=200); plt.close(fig)
print("[saved] figures 5-7 to", FIG_DIR)